In [ ]:
# Q4 Adversarial demo — Setup
import sys
from pathlib import Path
import random
SEED = 123
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
import torch
import matplotlib.pyplot as plt
from utils.utils import make_fig_name, save_figure, save_asset_manifest

IMAGES_DIR = Path('../q4_adversarial/images').resolve()
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
print('Images saved to', IMAGES_DIR)


In [ ]:
# 1) Build a small CNN and dataset (toy) and demonstrate FGSM/PGD
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms

# Tiny dataset: create 100 random images and binary labels
N = 100
X = torch.rand(N,3,32,32)
y = (torch.rand(N) > 0.5).long()

class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(), nn.Linear(32,2)
        )
    def forward(self,x):
        return self.net(x)

model = TinyCNN()
loss_fn = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr=1e-3)

# Quick training loop (1 epoch)
model.train()
for i in range(10):
    idx = torch.randint(0,N,(16,))
    xb = X[idx]
    yb = y[idx]
    logits = model(xb)
    loss = loss_fn(logits, yb)
    opt.zero_grad(); loss.backward(); opt.step()

# select 4 samples and make FGSM examples
def fgsm(images, labels, eps=0.03):
    images = images.clone().detach().requires_grad_(True)
    outputs = model(images)
    loss = loss_fn(outputs, labels)
    model.zero_grad(); loss.backward()
    pert = images + eps*images.grad.sign()
    return torch.clamp(pert,0,1)

samples = X[:4]
labels = y[:4]
adv = fgsm(samples, labels, eps=0.1)

# Save side-by-side original and adv images
import torchvision.utils as vutils
concat = torch.cat([samples, adv], dim=0)
grid = vutils.make_grid(concat, nrow=4, padding=2)
fig, ax = plt.subplots(figsize=(6,3))
ax.imshow(grid.permute(1,2,0).numpy())
ax.axis('off')
fig_path = make_fig_name('adversarial','samples','fgsm-eps0.1', ext='png', images_dir=str(IMAGES_DIR))
save_figure(fig, fig_path)
manifest = [{'filename': fig_path, 'width_in':6.0, 'height_in':3.0, 'dpi':300, 'caption_placeholder':'Original (top) and FGSM perturbed (bottom) samples'}]
print('Saved adversarial sample grid to', fig_path)


In [ ]:
# Save manifest and export assets
save_asset_manifest(manifest, str(IMAGES_DIR))
import pandas as pd
df = pd.DataFrame({'asset':[m['filename'] for m in manifest], 'caption':[m['caption_placeholder'] for m in manifest]})
df.to_csv(IMAGES_DIR / 'table-adversarial-assets.csv', index=False)
df.to_latex(IMAGES_DIR / 'table-adversarial-assets.tex', index=False)
print('Saved manifest and table to', IMAGES_DIR)
